<a href="https://colab.research.google.com/github/ANNYM25/python-together/blob/main/08-machine-learning/07_neural_net_from_scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 7 · A neural network from scratch

This is the big one. A neural network is just: multiply by weights, add a bias,
bend through a non-linear function, repeat — then adjust all the weights with
gradient descent (lesson 2) using the chain rule (the calculus track). We'll
build a **one-hidden-layer** net in pure NumPy that learns a curved boundary a
straight line could never fit.

In [ ]:
import numpy as np

## A problem a line can't solve

Label is 1 when `x0² + x1 > 0.7`. That boundary is **curved**, so linear models
fail — we need the non-linearity a hidden layer provides.

In [ ]:
rng = np.random.default_rng(6)
X = rng.normal(size=(80, 2))
y = (X[:, 0] ** 2 + X[:, 1] > 0.7).astype(float)[:, None]
print("X:", X.shape, " y:", y.shape, " positives:", int(y.sum()))

## The pieces

- **ReLU** is the non-linearity: `max(0, z)`. Without it, stacking layers would
  collapse back into one straight line.
- **Sigmoid** squashes the final number into a 0–1 probability.
- Weights `w1, w2` and biases `b1, b2` are what the network *learns*.

In [ ]:
def relu(z):
    return np.maximum(0, z)

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

hidden = 8
w1 = rng.normal(scale=0.4, size=(2, hidden)); b1 = np.zeros((1, hidden))
w2 = rng.normal(scale=0.4, size=(hidden, 1)); b2 = np.zeros((1, 1))

## The training loop: forward, loss, backward, step

Each iteration:
1. **Forward** — push `X` through the layers to get a prediction.
2. **Loss** — how wrong (mean squared error here).
3. **Backward** — the chain rule pushes the error back to every weight (this is
   *backpropagation*).
4. **Step** — nudge every weight downhill.

In [ ]:
lr = 0.15
for step in range(401):
    # forward
    z1 = X @ w1 + b1
    h = relu(z1)
    logit = h @ w2 + b2
    pred = sigmoid(logit)

    # loss
    loss = np.mean((pred - y) ** 2)

    # backward (chain rule)
    dpred = 2 * (pred - y) / len(X)
    dlogit = dpred * pred * (1 - pred)          # through sigmoid
    dw2 = h.T @ dlogit
    db2 = dlogit.sum(axis=0, keepdims=True)
    dh = dlogit @ w2.T
    dz1 = dh * (z1 > 0)                          # through relu
    dw1 = X.T @ dz1
    db1 = dz1.sum(axis=0, keepdims=True)

    # step
    for param, grad in [(w1, dw1), (b1, db1), (w2, dw2), (b2, db2)]:
        param -= lr * grad

    if step % 100 == 0:
        print(f"step {step:3d}  loss {loss:.4f}")

acc = float(np.mean((pred > 0.5) == y))
print(f"\ntrain accuracy: {acc:.3f}  (a straight line would be stuck near chance)")

That curved boundary getting learned is the entire principle behind deep
learning. Stack more of these layers, feed images or text, and you have the
machinery under CNNs (lesson 8) and the LLM project in `10-projects/`.

## 🏋️ Your turn

Hold out the **last 20 rows** as a test set, train on the first 60, and print
**test** accuracy. Then bump `hidden` from 8 to 32 and retrain — does it help,
or does it start overfitting the 60 training rows?

In [ ]:
X_train, X_test = X[:60], X[60:]
y_train, y_test = y[:60], y[60:]
# re-run the loop with X_train/y_train, then evaluate on X_test/y_test

### ✅ What you learned
- A neural net = linear steps + non-linear activations (ReLU), stacked.
- **Forward** computes a prediction; **backprop** sends the error back to weights.
- Training is still gradient descent — just with many parameters.
- Non-linearity lets it learn curved boundaries linear models can't.

**Next:** `08_cnn_intro.py` — specialised networks for images.